# BdSLW60 - Confidence intervals and paired tests on the official test signers

In [1]:
# ---- Cell 1: find every saved prediction ----
import re, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
INPUT_ROOT = Path('/kaggle/input'); OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
TEST_SIGNERS = (4, 8)
N_BOOT = 10000
rng = np.random.default_rng(0)

metas = sorted(INPUT_ROOT.rglob('meta.csv')); ys = sorted(INPUT_ROOT.rglob('y.npy'))
assert metas and ys, 'attach at least one notebook output that contains bdslw60_w2/meta.csv and y.npy'
meta = pd.read_csv(metas[0]); y = np.load(ys[0])
for p in ys[1:]:                       # every notebook must have built the same arrays
    y2 = np.load(p)
    if len(y2) != len(y) or (y2 != y).any():
        warnings.warn('labels differ in %s: predictions are checked against their own labels below' % p)
TEST = np.flatnonzero(meta.user.isin(TEST_SIGNERS).values)            # order of p1['test'] and of logits_test_*.npy
TEST_SORTED = np.concatenate([np.sort(np.flatnonzero(meta.user.values == u)) for u in TEST_SIGNERS])  # order of tta_probs
pos = {v: k for k, v in enumerate(TEST)}
signer = meta.user.values[TEST]
yt = y[TEST]
assert len(TEST) == 1276, len(TEST)
print('test clips: %d (U4 %d, U8 %d)' % (len(TEST), (signer == 4).sum(), (signer == 8).sum()))

PRED = {}    # key -> {seed: predicted class per clip, in TEST order}
PROB = {}    # key -> {seed: (1276, 60) probabilities, in TEST order} where available
def add(key, seed, pred=None, prob=None, src=''):
    if prob is not None:
        prob = np.asarray(prob, np.float64); pred = prob.argmax(1)
        PROB.setdefault(key, {})[seed] = prob
    if seed in PRED.get(key, {}) and (PRED[key][seed] != pred).any():
        warnings.warn('%s seed %d found twice with different predictions; keeping the first (%s ignored)' % (key, seed, src)); return
    PRED.setdefault(key, {})[seed] = pred

# 1) predtest_<key>_seed<s>.csv (idx, pred, true)
for p in sorted(INPUT_ROOT.rglob('predtest_*_seed*.csv')):
    m = re.match(r'predtest_(.+)_seed(\d+)\.csv$', p.name)
    if not m or '_LOSO_' in m[1]:
        continue
    d = pd.read_csv(p)
    if len(d) != len(TEST) or set(d.idx) != set(TEST):
        continue
    if (d.true.values != y[d.idx.values]).any():
        warnings.warn('labels in %s do not match y.npy; skipped' % p); continue
    pred = np.empty(len(TEST), int); pred[[pos[i] for i in d.idx]] = d.pred.values
    add(m[1], int(m[2]), pred=pred, src=p)

# 2) logits_test_<key>_seed<s>.npy (multi-stream notebook, TEST order)
for p in sorted(INPUT_ROOT.rglob('logits_test_*_seed*.npy')):
    m = re.match(r'logits_test_(.+)_seed(\d+)\.npy$', p.name)
    L = np.load(p)
    if L.shape != (len(TEST), 60):
        continue
    P = np.exp(L - L.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    add('MS_' + m[1], int(m[2]), prob=P, src=p)

# 3) tta_probs/P1_<stream>_seed<s>_<cfg>_<mode>.pt (compact ST-GCN streams, TEST_SORTED order)
try:
    import torch
except ImportError:
    torch = None
reorder = np.array([pos[v] for v in TEST_SORTED])
for p in sorted(INPUT_ROOT.rglob('P1_*_seed*_*.pt')):
    m = re.match(r'P1_(joint_motion|bone_motion|joint|bone)_seed(\d+)_(.+)_(inductive|transductive)\.pt$', p.name)
    if not m or torch is None:
        continue
    P = torch.load(p, map_location='cpu').numpy()
    Q = np.empty_like(P); Q[reorder] = P
    add('C_%s_%s_%s' % (m[1], m[3], m[4]), int(m[2]), prob=Q, src=p)

# 4) four-stream ensembles: equal-weight average of the stream probabilities, per seed
STREAMS = ('joint', 'bone', 'joint_motion', 'bone_motion')
cfgs = sorted({k.split('_', 2)[2] for k in PROB if k.startswith('C_joint_') and not k.startswith('C_joint_motion')})
for cfg in cfgs:
    ks = ['C_%s_%s' % (s, cfg) for s in STREAMS]
    if all(k in PROB for k in ks):
        for seed in set.intersection(*[set(PROB[k]) for k in ks]):
            add('ENS4_compact_' + cfg, seed, prob=sum(PROB[k][seed] for k in ks) / 4)
ks = ['MS_S0_slgcn'] + ['MS_S0_slgcn_' + s for s in STREAMS[1:]]
if all(k in PROB for k in ks):
    for seed in set.intersection(*[set(PROB[k]) for k in ks]):
        add('ENS4_slgcn', seed, prob=sum(PROB[k][seed] for k in ks) / 4)

print('\n%d prediction sets found:' % len(PRED))
for k in sorted(PRED):
    accs = [100 * (PRED[k][s] == yt).mean() for s in sorted(PRED[k])]
    print('  %-55s seeds %-9s top-1 %s' % (k, sorted(PRED[k]), ' '.join('%.2f' % a for a in accs)))

/tmp/ipykernel_17/89426332.py:17: UserWarning: labels differ in /kaggle/input/notebooks/rishta19/bdslw60-week-1/bdslw60_cache/y.npy: predictions are checked against their own labels below
  warnings.warn('labels differ in %s: predictions are checked against their own labels below' % p)
/tmp/ipykernel_17/89426332.py:17: UserWarning: labels differ in /kaggle/input/notebooks/rishta19/bdslw60-week-1-v2/bdslw60_cache/y.npy: predictions are checked against their own labels below
  warnings.warn('labels differ in %s: predictions are checked against their own labels below' % p)


test clips: 1276 (U4 637, U8 639)


/tmp/ipykernel_17/89426332.py:33: UserWarning: S0_slgcn seed 0 found twice with different predictions; keeping the first (/kaggle/input/notebooks/rishta123/bdslw60-multi-stream-ensemble/bdslw60_w2/predtest_S0_slgcn_seed0.csv ignored)
  warnings.warn('%s seed %d found twice with different predictions; keeping the first (%s ignored)' % (key, seed, src)); return
/tmp/ipykernel_17/89426332.py:33: UserWarning: S0_slgcn seed 1 found twice with different predictions; keeping the first (/kaggle/input/notebooks/rishta123/bdslw60-multi-stream-ensemble/bdslw60_w2/predtest_S0_slgcn_seed1.csv ignored)
  warnings.warn('%s seed %d found twice with different predictions; keeping the first (%s ignored)' % (key, seed, src)); return
/tmp/ipykernel_17/89426332.py:33: UserWarning: S0_slgcn seed 2 found twice with different predictions; keeping the first (/kaggle/input/notebooks/rishta123/bdslw60-multi-stream-ensemble/bdslw60_w2/predtest_S0_slgcn_seed2.csv ignored)
  warnings.warn('%s seed %d found twice wi


57 prediction sets found:
  BiLSTM_P1_adamw                                         seeds [0, 1, 2] top-1 83.46 82.99 84.01
  BiLSTM_P1_sgd                                           seeds [0, 1, 2] top-1 87.23 87.30 86.05
  BiLSTM_base_P1                                          seeds [0, 1, 2] top-1 76.72 81.58 80.88
  BiLSTM_base_P1_adamw                                    seeds [0, 1, 2] top-1 84.48 86.13 86.83
  BiLSTM_base_P1_sgd                                      seeds [0, 1, 2] top-1 87.77 89.11 88.24
  C_bone_bn_a1.0_inductive                                seeds [0, 1, 2] top-1 90.20 88.95 90.13
  C_bone_motion_bn_a1.0_inductive                         seeds [0, 1, 2] top-1 87.07 88.01 88.24
  C_bone_motion_none_inductive                            seeds [0, 1, 2] top-1 86.76 88.56 86.60
  C_bone_motion_self_a0.5_nobal_e30_inductive             seeds [0, 1, 2] top-1 89.81 90.52 88.95
  C_bone_motion_self_a0.5_nobal_e30_transductive          seeds [0, 1, 2] top-1 90.52 91.14

In [2]:
# ---- Cell 2: bootstrap machinery (stratified by test signer) and McNemar ----
strata = [np.flatnonzero(signer == u) for u in TEST_SIGNERS]
BOOT = np.concatenate([s[rng.integers(0, len(s), (N_BOOT, len(s)))] for s in strata], axis=1)   # (N_BOOT, 1276)

def correct(key):
    return np.stack([PRED[key][s] == yt for s in sorted(PRED[key])]).astype(np.float32)   # (seeds, clips)

def boot_acc(key):
    C = correct(key)
    point = 100 * C.mean()
    bs = 100 * C.mean(0)[BOOT].mean(1)          # seeds averaged per clip, then clips resampled
    return point, np.percentile(bs, [2.5, 97.5])

def mcnemar(a, b):
    """exact two-sided McNemar on two vectors of per-clip correctness"""
    n01 = int(((a == 1) & (b == 0)).sum()); n10 = int(((a == 0) & (b == 1)).sum())
    p = stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return n01, n10, p

def compare(key_a, key_b):
    A, B = correct(key_a), correct(key_b)
    d = 100 * (A.mean() - B.mean())
    bs = 100 * (A.mean(0)[BOOT].mean(1) - B.mean(0)[BOOT].mean(1))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    sa, sb = sorted(PRED[key_a]), sorted(PRED[key_b])
    common = [s for s in sa if s in sb]
    pairs = [(s, s) for s in common] if common else list(zip(sa, sb))
    mc = [mcnemar(A[sa.index(i)], B[sb.index(j)]) for i, j in pairs]
    per_signer = {u: 100 * (A.mean(0)[signer == u].mean() - B.mean(0)[signer == u].mean()) for u in TEST_SIGNERS}
    return dict(diff=d, ci_lo=lo, ci_hi=hi, p_boot_two_sided=2 * min((bs <= 0).mean(), (bs >= 0).mean()),
                mcnemar_p_per_seed=' '.join('%.3g' % m[2] for m in mc),
                a_only_b_only=' '.join('%d/%d' % (m[0], m[1]) for m in mc),
                seeds_significant='%d/%d' % (sum(m[2] < 0.05 for m in mc), len(mc)),
                dU4=per_signer[4], dU8=per_signer[8])
print('bootstrap ready:', BOOT.shape)

bootstrap ready: (10000, 1276)


In [3]:
# ---- Cell 3: 95% CI for every model in the main table ----
# Display names for the paper. Keys that are not found are skipped; add or rename keys here if cell 1 lists them differently.
MAIN = [
    ("Attn. Bi-LSTM, authors' recipe", 'BiLSTM_base_P1'),
    ('Attn. Bi-LSTM (AdamW)', 'BiLSTM_base_P1_adamw'),
    ('Attn. Bi-LSTM (SGD, our recipe)', 'BiLSTM_base_P1_sgd'),
    ('LSTM (SGD)', 'LSTM_P1_sgd'), ('BiLSTM (SGD)', 'BiLSTM_P1_sgd'), ('GRU (AdamW)', 'GRU_P1_adamw'),
    ('TCN (SGD)', 'TCN_P1_sgd'), ('MLP (AdamW)', 'MLP_P1_adamw'), ('Transformer (AdamW)', 'Transformer_P1_adamw'),
    ('SPOTER (AdamW)', 'SPOTER_P1_adamw'),
    ('Compact ST-GCN', 'STGCN_compact'), ('SL-GCN (S0)', 'S0_slgcn'), ('SL-GCN + missing-aware (S2)', 'S2_missing_aware'),
    ('CTR-GCN', 'CTRGCN'), ('2s-AGCN, joint', 'AGCN'),
    ('Compact ST-GCN, 4 streams', 'ENS4_compact_none_inductive'),
    ('SL-GCN, 4 streams', 'ENS4_slgcn'),
    ('Compact ST-GCN, 4 streams + AdaBN', 'ENS4_compact_bn_a1.0_inductive'),
    ('Compact ST-GCN, 4 streams + TENT', 'ENS4_compact_tent_inductive'),
    ('Compact ST-GCN, 4 streams + ours', 'ENS4_compact_self_a0.5_nobal_e30_inductive'),
]
rows = []
for label, key in MAIN:
    if key not in PRED:
        print('not found, skipped:', label, '(%s)' % key); continue
    pt, (lo, hi) = boot_acc(key)
    rows.append(dict(model=label, key=key, seeds=len(PRED[key]), top1=pt, ci_lo=lo, ci_hi=hi,
                     latex='%.1f [%.1f, %.1f]' % (pt, lo, hi)))
T = pd.DataFrame(rows)
print(T.round(2).to_string(index=False))
T.to_csv(OUT / 'stats_main_ci.csv', index=False)

not found, skipped: CTR-GCN (CTRGCN)
not found, skipped: 2s-AGCN, joint (AGCN)
                            model                                        key  seeds      top1  ci_lo  ci_hi             latex
   Attn. Bi-LSTM, authors' recipe                             BiLSTM_base_P1      3 79.730003  77.85  81.53 79.7 [77.8, 81.5]
            Attn. Bi-LSTM (AdamW)                       BiLSTM_base_P1_adamw      3 85.820000  84.12  87.49 85.8 [84.1, 87.5]
  Attn. Bi-LSTM (SGD, our recipe)                         BiLSTM_base_P1_sgd      3 88.379997  86.76  89.92 88.4 [86.8, 89.9]
                       LSTM (SGD)                                LSTM_P1_sgd      3 85.160004  83.36  86.89 85.2 [83.4, 86.9]
                     BiLSTM (SGD)                              BiLSTM_P1_sgd      3 86.860001  85.16  88.48 86.9 [85.2, 88.5]
                      GRU (AdamW)                               GRU_P1_adamw      3 86.989998  85.29  88.64 87.0 [85.3, 88.6]
                        TCN (SGD)      

In [4]:
# ---- Cell 4: paired comparisons ----
OURS = 'ENS4_compact_self_a0.5_nobal_e30_inductive'
PAIRS = [
    ('Ours vs no adaptation (4-stream compact)', OURS, 'ENS4_compact_none_inductive'),
    ('Ours vs TENT (4-stream compact)', OURS, 'ENS4_compact_tent_inductive'),
    ('Ours vs AdaBN (4-stream compact)', OURS, 'ENS4_compact_bn_a1.0_inductive'),
    ('Ours vs TENT (joint stream)', 'C_joint_self_a0.5_nobal_e30_inductive', 'C_joint_tent_inductive'),
    ('4-stream compact vs single compact ST-GCN', 'ENS4_compact_none_inductive', 'STGCN_compact'),
    ('SL-GCN 4-stream vs compact 4-stream', 'ENS4_slgcn', 'ENS4_compact_none_inductive'),
    ('2s-AGCN vs compact ST-GCN', 'AGCN', 'STGCN_compact'),
    ('Compact ST-GCN vs Transformer', 'STGCN_compact', 'Transformer_P1_adamw'),
    ('S2 vs S0 (missing-aware convolution)', 'S2_missing_aware', 'S0_slgcn'),
    ("Bi-LSTM: our recipe vs authors' recipe", 'BiLSTM_base_P1_sgd', 'BiLSTM_base_P1'),
]
rows = []
for label, a, b in PAIRS:
    miss = [k for k in (a, b) if k not in PRED]
    if miss:
        print('not found, skipped:', label, miss); continue
    rows.append(dict(comparison=label, **compare(a, b)))
C = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(C.round(3).to_string(index=False))
C.to_csv(OUT / 'stats_pairs.csv', index=False)
print('\ndiff: top-1 of the first minus the second (points, seeds averaged); CI: paired stratified bootstrap;'
      '\nMcNemar: exact, per seed (clips only the first got right / only the second got right).')

not found, skipped: 2s-AGCN vs compact ST-GCN ['AGCN']
                               comparison   diff  ci_lo  ci_hi  p_boot_two_sided         mcnemar_p_per_seed        a_only_b_only seeds_significant    dU4    dU8
 Ours vs no adaptation (4-stream compact)  1.776  1.097  2.482             0.000  6.04e-07 0.000431 0.00813       34/4 32/9 22/7               3/3  2.512  1.043
          Ours vs TENT (4-stream compact)  0.496 -0.000  0.993             0.059             0.0501 1 0.263       22/10 9/8 13/7               0/3  0.890  0.104
         Ours vs AdaBN (4-stream compact)  0.653  0.026  1.332             0.045              0.0139 1 0.15    27/11 14/14 20/11               1/3  1.204  0.104
              Ours vs TENT (joint stream) -0.392 -1.097  0.313             0.298           0.134 0.511 0.28    21/33 21/16 17/25               0/3  0.576 -1.356
4-stream compact vs single compact ST-GCN  3.579  2.534  4.650             0.000 8.91e-07 4.71e-06 1.81e-10    60/17 58/18 65/11            

In [5]:
# ---- Cell 5: sentences for the paper ----
for r in C.itertuples():
    print('%s: %+.1f points (95%% CI %+.1f to %+.1f); McNemar p per seed %s.'
          % (r.comparison, r.diff, r.ci_lo, r.ci_hi, r.mcnemar_p_per_seed))

Ours vs no adaptation (4-stream compact): +1.8 points (95% CI +1.1 to +2.5); McNemar p per seed 6.04e-07 0.000431 0.00813.
Ours vs TENT (4-stream compact): +0.5 points (95% CI -0.0 to +1.0); McNemar p per seed 0.0501 1 0.263.
Ours vs AdaBN (4-stream compact): +0.7 points (95% CI +0.0 to +1.3); McNemar p per seed 0.0139 1 0.15.
Ours vs TENT (joint stream): -0.4 points (95% CI -1.1 to +0.3); McNemar p per seed 0.134 0.511 0.28.
4-stream compact vs single compact ST-GCN: +3.6 points (95% CI +2.5 to +4.6); McNemar p per seed 8.91e-07 4.71e-06 1.81e-10.
SL-GCN 4-stream vs compact 4-stream: +1.4 points (95% CI +0.5 to +2.2); McNemar p per seed 0.000128 0.0444.
Compact ST-GCN vs Transformer: +0.4 points (95% CI -0.9 to +1.6); McNemar p per seed 0.779 0.569 0.355.
S2 vs S0 (missing-aware convolution): +0.2 points (95% CI -0.6 to +1.0); McNemar p per seed 0.098 0.899 0.603.
Bi-LSTM: our recipe vs authors' recipe: +8.6 points (95% CI +7.3 to +10.1); McNemar p per seed 8.61e-28 8.42e-15 2.16e-14.